# Workbook 2: Wrangling Data
## Summary
Data wrangling refers to all the steps it takes to prepare a data set for analysis. Each project may require different types of wrangling in order to prepare the data set depending on the data and/or the goal of the project. For a project where the goal is to wrangle data in a reproducible way, the wrangling procedures will be quite general and comprehensive to incorporate future needs.

This workbook will cover the following wrangling procedures:
- Loading
- Discovery
- Parsing 
- Recoding
- Duplicates
- Outliers
- Incorrect cases
- Missing cases
- Filtering
- Reshaping
- Merging
    



## Loading
In this workbook, we will be wrangling a data set, coffee.csv which was downloaded from kaggle.com and contains data that were scraped from [www.coffeereview.com](www.coffeereview.com).

One way to download this data set is by using a the direct file path. You can find the path using the following:
- on a Mac open Finder and find your file. Right click on the file and select "Get Info" 
- on Windows open Windows Explorer and find your file. Right click on the file and select "Properties"

In Project Focus, we will use an in-house package start.py from the library to call data. This file contains global paths to commonly used directories. They are set as global paths so they are not fixed to a specific user. Paths to the same SharePoint folder will look slightly different because it will contain each person's user credentials.

For example, these two paths are referencing the same folder:

"/Users/bah17005/OneDrive - University of Connecticut/project focus/focus_main/"
"/Users/kla21002/OneDrive - University of Connecticut/project focus/focus_main/"

The paths look different because they are being accessed from different computers. Further, the Project Focus folder is being accessed with a shortcut meaning each person creates a link in their own OneDrive folder that points to the SharePoint folder. This shortcut can be placed anywhere in a user's person OneDrive folder structure. So, the paths can be quite different depending on where a person puts the shortcut in OneDrive.

The way paths are created in start.py allows code to run without needing to account for the computer accessing the folder or a person's OneDrive folder structure.  

In [60]:
# install packages
# in terminal, run the following
# pip install pandas OR
# pip3 install pandas

# import packages
import os
import sys

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf


# using from assumes that the folder (e.g., library) is contained in the same working directory as the current file
# since we're referencing a library outside of our current working directory, we need to supply a path to the library before we import files from it
# 1. set the path to the folder above where library is located
# 2. from the library folder, import a python file called start
sys.path.insert(1, os.getcwd().split("code")[0] + "code/python/") # 1
from library import start

The file, start.py makes use of the os package, which imports a module to interact with the operating system. This allows us to perform tasks such as changing or removing a file, defining paths, etc.

| Function | Definition|  
| --- | --- | 
| os.getcwd() | get the current working directory (cwd) |
| os.chdir(x) | change the cwd to x |
| os.mkdir(p + x) | create a new folder called x in the p path |
| os.listdir(x) | list the folders and files in the x directory |



The directory is specified by first getting the current working directory (cwd), and then splitting the path of the cwd on the word "code." This word is used because the code folder is one that will be located in the same place for every user regardless of computer or OneDrive folder structure. The parent directory of the code folder is the main focus folder, so using this term to split on also retains the portion of the cwd path to the main focus folder. Everything before code will be used to generate user specific paths i.e., that portion of the path can vary between users.  

After running library import start, all the directories can be called in the same functions from other packages are called i.e., package.object

In [2]:
path = start.PYTHON_DIR + "python_on_boarding/"
file = "coffee.csv"

In [3]:
# read in the data
coffee = pd.read_csv(path + file)

# recast to a data frame
coffee = pd.DataFrame(coffee)

# create a text file to view the data
# this keeps the output from being truncated
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

# write the file
with open("coffee.txt", "w") as text_file:
    print(coffee, file = text_file)


## Discovery 

Now that the data has been read into Python, we'll start discovery to familiarize ourselves with the data. This includes checking the number of rows and columns, each column type, printing some cases, etc.

In [ ]:
coffee = pd.DataFrame(coffee)

print(coffee.shape)

# index coffee.shape to get the number of rows and columns
print("n rows:", coffee.shape[0])
print("n columns:", coffee.shape[1])

In [ ]:
# check the names of each column
list(coffee.columns)

In [ ]:
# check the data type for each of the 37 columns
coffee.dtypes

In [ ]:
# print cases
coffee.head() # by default this prints the first 5 rows
coffee.head(10) # print 10 rows

In [ ]:
coffee.tail() # by default this prints the last 5 rows
coffee.tail(10) # print the last 10 rows

We can also get the frequency of observations for specific variables.

In the coffee data, there is a variable for aroma, acidity, body, flavor, and aftertaste each of which are rated from 1 - 10 which according to the website, reflects how intense and pleasing each characteristic is.

We will take a look at the scores for aroma. Aroma is defined by "How intense and pleasurable is the aroma when the nose first descends over the cup and is enveloped by fragrance? Aroma also provides a subtle introduction to various nuances of acidity, taste and flavor: bitter and sweet tones, fruit, flower or herbal notes, and the like."

In [ ]:
# frequency of aroma
print(coffee.aroma.value_counts()) # the default prints the counts for the variable aroma
# by default these are sorted from highest count to lowest

# sort_values
# to print from the lowest to the higest count
print(coffee.aroma.value_counts().sort_values(ascending = True))

In [ ]:
# sort_index
# we can also print based on the response values (aka the index)
# this gives us an initial sense of the distribution. when we run our descriptives, we'll calculate the skewness and kurtosis
print(coffee.aroma.value_counts().sort_index(ascending = False))

We can extract these value counts to use them for additional calculations. For example, most coffees received an aroma score of 9, there were a handful that scored 10 so, we might be interested in seeing what percentage of coffees received an aroma score of 10 vs what percent received a 9.

In [ ]:
# by creating a list of the value counts, we can extract values
# index the specific values
aroma_freq10 = coffee.aroma.value_counts().sort_index(ascending = False).tolist()[0] # 10s
aroma_freq9 = coffee.aroma.value_counts().sort_index(ascending = False).tolist()[1] # 9s

# get the total count
aroma_freq_tot = sum(coffee.aroma.value_counts().sort_index(ascending = False).tolist())

# percentage of 10s
print("aroma % of 10s:", (aroma_freq10/aroma_freq_tot)*100)
# percentage of 9s
print("aroma % of 9s:", (aroma_freq9/aroma_freq_tot)*100)

We can also perform value counts on categorical data like locations. Because there are quite a few locations let's save this as an object so we can sift through it. Use the code below to look at the location of the roaster.

In [ ]:
# get the freq of categorical data
# we can even create a data frame of the counts
coffee_location_freq = pd.DataFrame(coffee.location.value_counts())
#print(coffee_location_freq.head(10))

# rename the column
coffee_location_freq.columns = ["freq"]

# create a new variable with the location (when saving value_counts the row index is observation being counted)
coffee_location_freq["location"] = list(coffee_location_freq.index)

print(coffee_location_freq)

How many coffee blends come from roasters in Glendora, California?

In [ ]:
# .loc to locate the freq for location = "Glendora, California"
coffee_location_freq.freq.loc[coffee_location_freq.location == "Glendora, California"]

# we can use the index (aka row names) in the same way we did to look through a variable
coffee_location_freq.freq.loc[coffee_location_freq.index == "Glendora, California"]

The above code uses .loc to locate specific cases within text or numeric data. This finds any cases that exactly match the string we supplied ("Glendora, California"). We can also use other comparison operators (e.g.,  >, <, or !=). However, it can sometimes be useful to search for a piece of a string, i.e., a **substring**. We can use this with .str, for example, .str.find("x") looks for any cases that contain x.

How many coffee roasts come from San Diego, California? This time, use .str.find() to just search for the city name.

In [ ]:
# str.find returns the location of the substring within the string
# thus if a value is returned that is greater than or equal to 0, then the substring was found somewhere in the string
# any cases that return -1 did not find the substring in the string
coffee_location_freq.freq.loc[coffee_location_freq.index.str.find("San Diego") >= 0] # one case contains a typo

We see multiple cases matching "San Diego" because one of these cases contains a typo in "California." For now, we won't do anything about the typo, but we will eventually want to recode so that our value counts are accurate.

## Recoding
There are multiple reasons for recoding data. We may have reverse scored items on a survey scale or need to create dummy codes for categorical responses. 

For example, to include gender in a regression model, we would create a new variable for each gender (e.g., male, female, nonbinary) where each case receives a 1 or 0. Those who respond "male" are scored a 1 in the male variable while those who respond "female" or "nonbinary" are scored a 0. Those who respond "female" score a 1 in the female variable and those who respond nonbinary score a 1 in the nonbinary variable.

The coffereviews.com website notes that while the overall rating is scored from 1-100, they only qualitatively define certain clusters of scores:

|Rating|Label|Interpretation|
|---|---|---|
|>= 97|Exceptional|We have not tasted a coffee of this style as splendid as this one for a long, long time|
|95-96|Outstanding|Perfect in structure, flawless, and shockingly distinctive and beautiful|
|93-94|Great|Originality, beauty, individuality and distinction, with no significant negative issues whatsoever|
|91-92|Very Good|Coffee with excitement and distinction in aroma and flavor – or an exceptional coffee that still perhaps has some issue that some consumers may object to but others will love – a big, slightly imbalanced acidity, for example, or an overly lush fruit|
|89-90|Good|A very good coffee, drinkable, with considerable distinction and interest|
|87-88|Moderate|An interesting coffee but either 1) distinctive yet mildly flawed, or 2) solid but not exciting|
|85-86|Acceptable|An acceptable, solid coffee, but nothing exceptional — the best high-end supermarket whole bean, for example|
|80-84|Fair| |
|< 80|Poor| |

In [ ]:
# look at the minimum and maximum values
print("min:", coffee.rating.min())
print("max", coffee.rating.max())

When we look at the minimum and maximum ratingss, we see that they only range from 63 to 98. Even though, technically they can score from 1-100 raters are only using the upper portion of the scale which is likely influenced by the lack of construct definitions for anything below a score of 80. We also notice that no coffees received a 99 or 100 which again might be influenced by a lack of information about what distinguished a rating of 97 from a rating of 99.

Therefore, lets recode the rating variable to reflect its qualitative category.

In [ ]:
# define how to recode the variables
def rating_label(series):
    if series >= 97:
        return "exceptional"
    elif series >=95 and series <97:
        return "outstanding"
    elif series >=93 and series <95:
        return "great"
    elif series >=91 and series <93:
        return "very good"
    elif series >=89 and series <91:
        return "good"
    elif series >=87 and series <89:
        return "moderate"
    elif series >=85 and series <87:
        return "acceptable"
    elif series >=80 and series <85:
        return "fair"
    elif series <80:
        return "poor"
    elif series.isna() == True:
        return None

coffee["rating_label"] = coffee["rating"].apply(rating_label) # apply the recode

print(coffee[["rating", "rating_label"]][1:25]) # check that the recode worked


In [ ]:
# define how to recode the variables
def rating_rank(series):
    if series >= 97:
        return 1
    elif series >=95 and series <97:
        return 2
    elif series >=93 and series <95:
        return 3
    elif series >=91 and series <93:
        return 4
    elif series >=89 and series <91:
        return 5
    elif series >=87 and series <89:
        return 6
    elif series >=85 and series <87:
        return 7
    elif series >=80 and series <85:
        return 9
    elif series <80:
        return 10
    elif series.isna() == True:
        return None

coffee["rating_rank"] = coffee["rating"].apply(rating_rank) # apply the recode

print(coffee[["rating", "rating_rank", "rating_label"]][1:25]) # check that the recode worked

What is the frequency of each category? Here we'll use the rank variable because when we try to sort with the qualitative categories, it will sort alphabetically, but we want to order them based on their ranking (i.e., highest scoring to lowest scoring). 

In [ ]:
coffee.rating_rank.value_counts().sort_index(ascending = True)

## Anomalous Cases
Another reason we might recode is to correct any anomalies in our data. These could be typos like the one we found when searching for San Diego. They could also include cases where there is clearly incorrect data. This would include a case where someone responded that their age was 2.

In [ ]:
print(coffee.location[coffee.location.str.find("Calfornia") >= 0]) # one case contains a typo

# recode this specific typo
coffee.location.str.replace("Calfornia", "California")

# check that the value count for San Diego is now 112 
print(coffee.location.value_counts())


With the city and state combined, it's difficult to catch any other typos because there are multiple combinations of cities and states. If we split the city and states up, we can sort the cases into alphabetically which will make it easier to flag any typos. When looking to split up data, we want to find a pattern to tell Python where to split. We can split by spaces or in this case, by commas.

In [ ]:
# split location
split_location = pd.DataFrame(coffee.location.str.split(pat = ",", expand = True))
split_location.head()

Notice that the location variable was split into 3 columns, meaning that for some cases there were 2 commas in the location variable.

Let's go through the value counts in each column to try to figure out why the data were split into 3 columns. These output will be long so we are going to print it to a text file called output.txt which we will use to look at each column of split_location.

In [21]:
# create an object called output
output = split_location[2].value_counts().sort_index(ascending = True)

# this keeps the output from being truncated when we print()
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

# create a file called output.txt and print output in it
with open("output.txt", "w") as text_file:
    print("split_location[2]: \n\n {} \n\n".format(output), file = text_file)

In [22]:
# we output to the same file so this will add to output.txt
output = split_location[1].value_counts().sort_index(ascending = True)

# add to the file called output.txt and print output in it
# replacing "w" with "a" appends output.txt instead of overwriting it
with open("output.txt", "a") as text_file:
    print("split_location[1]: \n\n {} \n\n".format(output), file = text_file)

In [23]:
output = split_location[0].value_counts().sort_index(ascending = True)

# add to the file called output.txt and print output in it
with open("output.txt", "a") as text_file:
    print("split_location[0]: \n\n {} \n\n".format(output), file = text_file)

There are some cases that do not follow a uniform method for reporting the location. Some locations outside of the US report City, State, Country while others report City, Country. Locations in the United States generally report City, State with the exception of Hawaii which reports City, County, State likely because the county also corresponds to the island. 

For example, consistent data would have from Hawaii all formatted in the same way so, Hawaii shouldn't appear in the second column (i.e., split_location[1]) but indeed these data are inconsistent so some cases include the Hawaiin roaster's county while others do not. 

We also want to check for spelling inconsistencies in all columns. For example, if we run split_location[1].value_counts().sort_index(ascending = True).tail(), we see that Washington is misspelled in one case.

Look through output.txt and in the code below we create a dictionary with most of the spelling errors and what they should be corrected to. There are a few cases where it was unclear if they were misspelled so, I flagged them for you to practice looking up. Use .str.find to search for the case and if any of them need to be recoded, add them to the dictionary.

In [ ]:
spell_check = {
    "Hawai'i" : "Hawaii", "Hawai’i" : "Hawaii", # split_location[2]
    "Californiaa" : "California", "Washingto" : "Washington", "Chia-yi" : "Chia-Yi", "Kauia" : "Kauai", "MInnesota" : "Minnesota", # split_location[1]
    "Chia-Yi City" : "Chia-Yi", "Chiya-Yi" : "Chia-Yi", "Kaohsiung City" : "Kaohsiung", "New Taipei City" : "New Taipei", "Sacamento" : "Sacramento",
    "Taichung City" : "Taichung", "Tainan City" : "Tainan", "Taipai" : "Taipei", "Taipei City" : "Taipei", "Taoyuan City" : "Taoyuan"
    } # create dictionary
print(spell_check)

# Branford Connecticut
# Maryville
# Marysville
# Mayville
# Zhuwei
# Zhubei

spell_check["Washingto"] # check that the dictionary worked

Next, lets use our dictionary to recode our variables.


In [25]:
# 1 initiate an empty list
# 2 for each case in the list 
# 3 create a variable called recode = the case
# 4 check if the case is found in the spell.check.keys aka if the word is a key in the dictionary
# 5 if the case is a key in the dictionary then it's misspelled so run the dictionary on it
# 6 overwrite recode = the spell_check result
# 7 add the object recode to the empty list
# 7 return the list

def spell_recode(list):
    list_recode = [] # 1
    for i in list: # 2
        recode = i # 3
        if i in spell_check.keys(): # 4
            recode = spell_check[i] # 5
        list_recode.append(recode) # 6
    return list_recode # 7

In [26]:
# 1 for each column i in split_location 
# 2 create a new variable name for the recode called recode + i
# 3 set the new variable equal to the recode of split_location column i


for i in split_location: # 1
    split_location["recode" + str(i)] = spell_recode(split_location[i]) # 2, 3

In [ ]:
print(split_location.head())
print(coffee.location.head())

## Merging
We can also compile data across multiple data frames using .merge(). 

In [28]:
# create a new variable with the recoded state variable
coffee["state"] = split_location["recode1"].str.strip()

In [ ]:
# create a new data frame of all the US states and territories
states = pd.DataFrame(["Alaska", "Alabama", "Arkansas", "American Samoa", "Arizona", 
"California", "Colorado", "Connecticut", "District of Columbia", "Delaware", 
"Florida", "Georgia", "Guam", "Hawaii", "Iowa", "Idaho", "Illinois", "Indiana", 
"Kansas", "Kentucky", "Louisiana", "Massachusetts", "Maryland", "Maine", "Michigan", 
"Minnesota", "Missouri", "Mississippi", "Montana", "North Carolina", "North Dakota", 
"Nebraska", "New Hampshire", "New Jersey", "New Mexico", "Nevada", "New York", "Ohio", 
"Oklahoma", "Oregon", "Pennsylvania", "Puerto Rico", "Rhode Island", "South Carolina", 
"South Dakota", "Tennessee", "Texas", "Utah", "Virginia", "Virgin Islands", "Vermont", 
"Washington", "Wisconsin", "West Virginia", "Wyoming"])

# rename the column states
us_states = states.rename(columns = {0: "name"})

# add a variable for country
us_states["country"] = "US"

print(us_states.head(5))

In [ ]:
print(type(us_states["name"]))
print(type(coffee["state"]))

To merge data we will assign one set of data to the left and the other to the right. In this case, the left data is coffee and right data is us_states. When looking at the code, the left data set is on the left of .merge so, in coffee.merge(us_states) coffee would be the left data set. 

The inputs you need to provide to run .merge() include:
|Input|Definition|
|---|---|
|**left_on** = ["l_var1", "l_var2"] | l_var is the name of the variable(s) from the left data set you would like to use to match|
|**right_on** = ["r_var1", "r_var2"] | l_var is the name of the variable(s) from the left data set you would like to use to match|
|**on** = ["var1", "var2"] | used insead of left_on and right_on if the matching variable(s) have the same name(s) in the left and right data|
|**how** = "left", "right", "inner", or "outer" | designate which data to retain in the merge (see more below)|
|**indicator** = "True" or "False"| if True, adds a column to the output DataFrame called “_merge” with information on the source of each row|

When determining what to set **how**  to:
- **inner**: use only keeps of cases that appear in both frames
- **left**: use only cases from left frame, which will keep all matches and any non-matches in the left data set
- **right**: use only cases from right frame, which will keep all matches and any non-matches in the right data set
- **outer**: keeps all cases from both data frames


In [ ]:
print(coffee.state.head(5))

In [32]:
# left: coffee
# match on the variable "state"

# right: us_states
# match on the variable "name"

# keep all the coffee data (aka all the left data)
coffee_us_states = coffee.merge(us_states, left_on = "state", right_on = "name", how = "left", indicator = True)

After we run a merge, we can output information about the merge by printing the .value_counts(). This will provide us with the number of cases were found in the left and right data frames as well as how many were found in both (combined). 

In [ ]:
# use the indicator from merge to get the value counts
print(coffee_us_states._merge.value_counts())

print("coffee_us_states rows:", coffee_us_states.shape[0])

## Duplicates
We can also check for exact and partial duplicates in our data. The function .duplicated() returns boolean data where True indicates that the case is a duplicate. In the coffee data, we might expect some names of the coffee blends to be duplicated because they may exist at multiple locations.

In [ ]:
print("name is duplicated: \n", coffee["name"].duplicated().value_counts(), "\n")

print("name + roaster is duplicated: \n", coffee[["name", "roaster"]].duplicated().value_counts(), "\n")

There were 94 cases in the coffee data frame whose combinations of names + roaster were duplicated. Now lets check if there are any exact duplicates (i.e., duplicated across every column)

In [ ]:
coffee.duplicated().value_counts()

There were no cases that were exact duplicated in coffee. If there were we could use .drop_duplicates to remove any duplicate cases. The input **keep** determines which duplicate case to retain and **subset** is an optional input that determines which variables should be considered for determining if a case is duplicated.

To practice, create a subset of data dropping any cases duplicated on "name" and "roaster".

In [ ]:
coffee_no_dups = coffee.drop_duplicates(keep = "first", subset = ["name", "roaster"])

print(coffee_no_dups.shape[0]) # check that this matches name + roaster is duplicated: False 

## Missing cases
In Python both None and NaN (not a number) are considered missing values. While None is considered its own type of data, NaN's are floats. Use .isna() and .notna() to detect missing values in a data frame. Each of these commands return boolean data; where with .isna(), a returned value of True indicates the data is missing. 

You can assign missing values using None or np.nan. 

When making calcuations with missing data, use the input skipna = True.

In [ ]:
a = None 
b = np.nan
print(type(a))
print(type(b))

In [ ]:
# check if the first 5 cases are missing
print(coffee.location.isna()[1:5], "\n")

# find missing data in specific variables
# print the number of missing cases in the flavor variable
print("flavor missing: \n", coffee.flavor.isna().value_counts(), "\n")

# print the number of missing cases in the rating variable
print("rating missing: \n", coffee.rating.isna().value_counts(), "\n")

In [ ]:
# find missing data across an entire data frame and sort descending (most missing to least missing)
print(coffee.isna().sum().sort_values(ascending = False))

### Handling missing data
While there are more advanced methods for missing data imputation, the two ways to handle missing data shown here are:
1. Replace missing values with the variable mean
2. Drop missing values from the data frame

In [ ]:
# method 1: fill missing value using mean
coffee['flavor'].fillna((coffee['flavor'].mean()), inplace= True)

# print the number of missing cases in the flavor variable
# check that the inputs worked
print("flavor missing: \n", coffee.flavor.isna().value_counts(), "\n")

In [ ]:
# create a subset of data
coffee_subset = coffee[["aftertaste", "flavor", "acid"]]

# check n missing before imputation
print("missing before imputation: \n", coffee_subset.isna().sum().sort_values(ascending = False), "\n") 
print("means before imputation: \n ", coffee_subset.mean(), "\n")

# impute mean for each column in the data frame
for i in coffee_subset:
    coffee_subset[i].fillna((coffee_subset[i].mean()), inplace= True)

# check missing data after imputation
print("missing after imputation: \n", coffee_subset.isna().sum().sort_values(ascending = False)) 
print("means after imputation: \n ", coffee_subset.mean(), "\n")


In [ ]:
# method 2 drop the rows with missing value
# reset the subset of data
coffee_subset = coffee[["aftertaste", "flavor", "acid"]]

# check n rows and columns before dropping missing data
print("n coffee_subset before drop: \n", coffee_subset.shape, "\n") 

# drop missing data
# axis: 0 = drop rows with missing data, 1 = drop columns with missing data
# how: "any" = drop if any missing cases are found, "all" = drop if all cases are missing
# inplace: create a new data frame or replace inplace of the current data frame
coffee_subset.dropna(axis = 0, how = "any", inplace = True) 

# check n rows and columns after drop
print("n coffee_subset after drop: \n", coffee_subset.shape, "\n") 

# find missing data across an entire data frame and sort descending (most missing to least missing)
print("missing data counts: \n", coffee_subset.isna().sum().sort_values(ascending = False))

## Filtering

Data filtering is the process of selecting a portion (or subset) of your data for viewing or analysis. Let's filter our data into a subset that just includes coffee roasters from San, Diego, California. 

In [ ]:
# filter 
coffee_sd_ca = coffee[coffee.location == "San Diego, California"]
print("n row filter San Diego, CA: \n", coffee_sd_ca.shape[0], "\n")

# filter using .loc
coffee_sd_ca = coffee.loc[coffee.location == "San Diego, California"]
print("n row filter San Diego, CA using .loc: \n", coffee_sd_ca.shape[0], "\n")


We can also use conditions to filter data, for example, we can create a subset of data that only contains cases that were rated above a 97 and therefore fell into the highest category, exceptional. 

In [ ]:
coffee_exceptional = coffee[coffee.rating >= 97]

print(coffee_exceptional[["rating", "rating_label"]])

We can also filter data that looks for patterns of a string. Some locations contained the pattern "City" e.g., "Jersey City" so, we can filter data to include all cases that detect the pattern "City" in the location variable.

In [ ]:
# filter cases - only include locations with "City"
coffee_city = coffee[coffee.location.str.contains('City')]

# look at the value counts for locations that contained the pattern "City"
print(coffee_city.location.value_counts())

## Descriptives

### Central tendency
Measures of central tendency (i.e., mean, median, and mode) provide a sense of the central position in that set of data. 

- **Mean**: the sum of all values divided by the total number of values.
- **Median**: the middle value in an ordered set of numbers.
- **Mode**: most frequent value.

In [ ]:
print("mean:", coffee.rating.mean())
print("median:", coffee.rating.median())
print("mode:", coffee.rating.mode())

### Deviation of scores
Measures of deviance (i.e., standard deviation and variance) provide a sense of how disperse a set of data is. Smaller scores indicate the set of data is more uniform because it is less spread out. Larger scores indicate the set of data is more variable because it is more spread out.
- **Variance**: the average of the squared differences from the mean
- **Standard deviation**: the square root of the variance, this value is standardized so it is in the same units as the mean

In [ ]:
# deviation from mean
print("variance:", coffee.rating.var())
print("sq rt variance:", np.sqrt(coffee.rating.var()))
print("st dev:", coffee.rating.std())

### Skewness and Kurtosis
**Skewness** measures the amount of symmetry in a set of data. Negatively skewed data have a concentration of high scores but are being pulled by low scoring outliers. Positive skewed data have a concentration of low scores that are being pulled by high scoring outliers.

**Kurtosis** measures the presence of outliers. A flatter distribution has a stronger presence of outliers. Kurtosis values are expected arount 3 with higher values indicating a higher peak in the data.

In [ ]:
print("skew:", coffee.rating.skew())
print("kurt:", coffee.rating.kurtosis())

In [ ]:
coffee.rating == 1

### Correlations
Correlations measure the amount of association between two variables (e.g., x and y). Negative correlations indicate as one variable (x) increases, the other (y) decreases. While positive correlations indicate the the variables increase and decrease together. 

The coffeereview.com site suggests that the different categories for aroma, acidity, body, flavor, and aftertaste do not contribute to the overall rating. Each category is scored 1-10 while the overall rating is scored 1-100. Let's see if there are some categories that are assosciated with the total score more than others. 

In [ ]:
# create a subset of variables to run correlations on
# outputs a correlation matrix
coffee[["rating", "aroma", "acid", "body", "flavor", "aftertaste"]].corr()

## Inferences
### Linear regression
Linear regression models a relationship between 1 or multiple predictors (x1, x2, x3) and an outcome (y). The regression coefficients indicate the amount of change in the outcome (y) associated with a 1-unit change in the predictor (x1) after accounting for the other predictors in the model (x2 & x3). 

Looking at the correlation matrix, it appears that the acidity of the coffee has the strongest relationship with the overall score (*r* = .82) with flavor, aroma, and aftertaste following. However, there is also information that suggests that smell and taste are interconnected. So, these correlations don't capture any potential interaction between these categories. 

The correlations also do not look at the effects after accounting for the other categories. Therefore, we'll see if we can use a linear regression model to predict the overall score based on the categories. We'll also add in an interaction between aroma and flavor to account for any interaction between taste and smell.

In [ ]:
# linear regression
x = coffee[["aroma", "acid", "body", "flavor", "aftertaste"]]
x = sm.add_constant(x)

y = coffee["rating"]

model = sm.OLS(y, x, missing = "drop")
res = model.fit()
print(res.summary())